# Training-free guidance (TFG) on Tahoe cell states

Comparison method 2 of 3 on Modality 2: the condition enters through the **sampling
trajectory**. The diffusion model runs unconditionally and is steered at every step by the
gradient of a fixed objective on the predicted clean cell state. Nothing is trained: the
diffusion model is the (drug, cell line)-conditional base shared by the cell notebooks, run
with its null condition, and the objective is a **per-drug Gaussian classifier** fitted by
moments on the training cells, the cell-state stand-in for a pretrained classifier. The target
is the drug.

**Paper:** Ye, Lin, Han, Xu, Liu, Liang, Ma, Zou, Ermon.
*TFG: Unified Training-Free Guidance for Diffusion Models.* NeurIPS 2024.
[arXiv:2409.15761](https://arxiv.org/abs/2409.15761)

**Official code exists** and was read:
[YWolfeee/Training-Free-Guidance](https://github.com/YWolfeee/Training-Free-Guidance),
`methods/tfg.py`. `src/guidance/baselines/tfg.py` re-expresses `guide_step` against our
DDIM sampler, with their schedules and their CIFAR-10 label-guidance hyperparameters as
defaults. Its header lists what was kept and what changed.

**Two scorers, deliberately different.** The Gaussian classifier guides; a k-nearest-neighbour
vote over real training cells, labelled with their true drug, scores. If one function did both,
"condition accuracy" would measure how well guidance games its own objective. A guidance
method has not been shown to work because its own objective went up.

Run top to bottom. On Colab set **Runtime > Change runtime type > GPU** first. `base.pt` from
the TC-LoRA or cross-attention cell notebook (same pair condition) can be uploaded to skip base
training.

## How this notebook implements TFG

**Notation.** $x_k$ is the noisy image at diffusion step $k$ ($K = 1000$; the network receives
$t = k/K$), $k'$ the next step of the $N = 50$-step DDIM schedule, $\bar\alpha_k$ the cosine
schedule, $\varepsilon_\theta$ the frozen MLP over the 50 principal components, $\varnothing$ the null condition
(drug and cell line both null). $y$ is the target drug and $p_r(y \mid x)$ a fixed classifier on clean
cell states: one Gaussian per drug with the drug's mean $\mu_y$ over training cells and a shared
covariance $\Sigma$,
$\log p_r(y \mid x) = \log\operatorname{softmax}_d\!\big(-\tfrac12 (x-\mu_d)^\top\Sigma^{-1}(x-\mu_d) + \log\pi_d\big)_y$.

**The paper's idea.** Steer an *unconditional* diffusion model with any differentiable objective
on clean data, without training anything: at every step, estimate the clean image, score it,
and move along the gradient. TFG shows that earlier training-free methods are special cases of
two terms: a *variance* term, the gradient with respect to the noisy state taken through the
denoiser, and a *mean* term, gradient steps taken on the clean estimate itself. Both act on a
smoothed objective, and both have a schedule over the trajectory. Per step $k \to k'$
(Algorithm 1):

$$\hat x_0 = \operatorname{clip}\!\Big(\frac{x_k - \sqrt{1-\bar\alpha_k}\;\varepsilon_\theta(x_k, k, \varnothing)}{\sqrt{\bar\alpha_k}}\Big),
\qquad
\log \tilde f(x) = \log \frac{1}{n_\varepsilon}\sum_{i=1}^{n_\varepsilon} \exp f\big(x + \sigma^{\mathrm s}_k\,\delta_i\big),\quad f(x) = \log p_r(y \mid x)$$

$$\Delta_k = \rho_k\, \nabla_{x_k} \log \tilde f\big(\hat x_0(x_k)\big)
\qquad\text{(variance guidance; the gradient goes through } \varepsilon_\theta)$$

$$\hat x_0^{(j+1)} = \hat x_0^{(j)} + \mu_k\, \nabla \log \tilde f\big(\hat x_0^{(j)}\big),\ \ j = 0,\dots,N_{\text{iter}}-1,
\qquad \Delta_0 = \hat x_0^{(N_{\text{iter}})} - \hat x_0
\qquad\text{(mean guidance)}$$

$$x_{k'} = \underbrace{\sqrt{\bar\alpha_{k'}}\,\hat x_0 + \sqrt{1-\bar\alpha_{k'}}\,\hat\varepsilon}_{\text{plain DDIM step, } \hat\varepsilon \text{ recomputed from the clipped } \hat x_0}
\;+\; \frac{\Delta_k}{\sqrt{\bar\alpha_k / \bar\alpha_{k'}}} \;+\; \sqrt{\bar\alpha_{k'}}\;\Delta_0$$

Schedules are theirs: $\rho_k = \rho\, a_k / \bar a$ and $\mu_k = \mu\, a_k / \bar a$ with
$a_k = \bar\alpha_k / \bar\alpha_{k'}$ and $\bar a$ its mean over the trajectory, so the mean
strength equals the $\rho$ and $\mu$ you set; $\sigma^{\mathrm s}_k = \sigma^{\mathrm s}\sqrt{1-\bar\alpha_k}$.
(The paper writes the smoothing scale $\sigma_t$; it is $\sigma^{\mathrm s}$ here so that $\sigma$
is not overloaded.)

**How it is used here.** The unconditional denoiser is the (drug, cell line)-conditional MLP run with
both ids null. The objective is $\log p_r(y \mid \hat x_0)$ from the Gaussian classifier above,
fitted by moments (means, one shared covariance, class frequencies) on the training cells: no
pretrained cell-state classifier exists and the project trains none. The hyperparameters are
their CIFAR-10 label-guidance script, unchanged. The DDIM step inside the guided sampler is the
same step the rest of this repository uses, and the first cell after setup asserts that at
$\rho = \mu = 0$ the guided sampler reproduces plain DDIM exactly. Nothing is trained. Each
guided step costs one denoiser forward, one backward, and $N_{\text{iter}} + 1$ classifier
evaluations; the notebook reports the counts rather than a single NFE.

| In the paper / official code | In this notebook |
|---|---|
| Alg. 1, clean estimate $\hat x_0$, clamped | `TFG._predict_x0` in `tfg.py` |
| Alg. 1, variance guidance $\Delta_k$, gradient through the denoiser | the `rho` block of `TFG.sample` |
| Alg. 1, mean guidance, $N_{\text{iter}}$ steps on the clean estimate | the `iter_steps` loop of `TFG.sample` |
| Alg. 1, the update $x_{k'}$; `_predict_x_prev_from_zero` recomputes $\hat\varepsilon$ from the clipped $\hat x_0$ | `TFG.sample`; `ddim_sample` here uses the same convention, and *Set up TFG* asserts the two agree at $\rho = \mu = 0$ |
| $\log\tilde f$, `tilde_get_guidance` | `TFG._log_f_tilde` |
| schedules `get_rho`, `get_mu`, `get_std` | `TFG._strength`, `TFG._std` |
| `ImageLabelGuidance`: the objective is a classifier's $\log p(y \mid \hat x_0)$ | `make_log_p` on the Gaussian classifier (`guide_logits`) |
| `scripts/cifar10_label.sh`: $\rho = 1$, $\mu = 0.25$, $\sigma^{\mathrm s} = 10^{-3}$, $n_\varepsilon = 1$, $N_{\text{iter}} = 4$, one recurrence, 50 steps | the `Cfg` defaults |
| an unconditional diffusion model | `eps_uncond`: our class-conditional model run with the null label |
| not in TFG: a second classifier for scoring, the wrong-target row, paired generators, the strength sweep | *Sample*, *Condition accuracy*, *Guidance-strength sweep*: our evaluation protocol |

**What the paper did that this notebook does not.** Their sampler is stochastic DDIM
($\eta = 1$); ours is deterministic ($\eta = 0$) to match the other arms, and the $\eta$ term is
implemented if you want theirs. Their diffusion model is an unconditional pretrained CIFAR-10
DDPM; ours is a conditional MLP standing in as unconditional through its null condition.
Their objective is an image classifier; ours is a Gaussian classifier on cell states. They
searched hyperparameters per task; we take their CIFAR-10 values as they are, so these settings
are not tuned for cell states. Six deviations are listed in the header of
`src/guidance/baselines/tfg.py`.

In [ ]:
# Colab bootstrap. Run once.
import importlib.util, subprocess, sys, os

for pkg in ("datasets",):
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

if importlib.util.find_spec("guidance") is None:
    if not os.path.exists("discrete-generative-models-proj-1"):
        subprocess.run(["git", "clone", "-q",
                        "https://github.com/imperorrp/discrete-generative-models-proj-1.git"],
                       check=True)
    sys.path.insert(0, "discrete-generative-models-proj-1/src")

import torch
from guidance.baselines.tfg import TFG, TFGConfig

def _git_sha():
    for d in ("discrete-generative-models-proj-1", "."):
        try:
            return subprocess.check_output(["git", "-C", d, "rev-parse", "--short", "HEAD"],
                                           text=True, stderr=subprocess.DEVNULL).strip()
        except Exception:
            pass
    return "unknown"

GIT_SHA = _git_sha()
print("code version:", GIT_SHA)
print("device:", "cuda" if torch.cuda.is_available()
      else "CPU  <-- Runtime > Change runtime type > GPU")

## Config

In [ ]:
import math, time, json
from dataclasses import dataclass, asdict, fields
import os

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

@dataclass
class Cfg:
    seed: int = 0                  # training / sampling randomness
    split_seed: int = 0            # shuffling only; the Tahoe split is fixed by the data files
    batch_size: int = 256
    base_epochs: int = 30
    lr_base: float = 2e-4
    K: int = 1000                  # diffusion training steps
    sample_steps: int = 250        # sampling steps, as the cell base's own notebook
    reuse_base: bool = True        # load base.pt if present instead of retraining
    limit: int = 0                 # >0 truncates the training set, for smoke tests
    samples_per_condition: int = 200
    sweep: tuple = (0.0, 0.5, 1.0, 2.0, 4.0)
    shrinkage: float = 1e-2        # added to the diagonal of the shared covariance
    condition: str = 'pair'        # pair (drug + cell line) | drug; see guidance.data.cells
    dose: float = -1.0             # -1 keeps every dose; 0.05, 0.5 or 5 keeps that dose (uM) plus controls
    # TFG -- defaults are the official CIFAR-10 label-guidance script
    rho: float = 1.0
    mu: float = 0.25
    sigma: float = 0.001
    eps_bsz: int = 1
    iter_steps: int = 4
    recur_steps: int = 1
    eta: float = 1.0               # ancestral DDPM, as the cell base's own notebook and TFG's own CIFAR-10 script

cfg = Cfg()

# Any field can be overridden from the environment, e.g. CFG_BASE_EPOCHS=1 CFG_LIMIT=5000, so
# headless runs (nbconvert, a server, a different GPU) never need a cell edited.
for _f in fields(Cfg):
    _v = os.environ.get(f"CFG_{_f.name.upper()}")
    if _v is not None:
        _cur = getattr(cfg, _f.name)
        setattr(cfg, _f.name, tuple(float(t) for t in _v.split(",")) if isinstance(_cur, tuple)
                else type(_cur)(_v) if not isinstance(_cur, bool) else _v.lower() in ("1", "true", "yes"))
        print(f"cfg.{_f.name} <- {getattr(cfg, _f.name)!r} (from CFG_{_f.name.upper()})")
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(cfg.seed); np.random.seed(cfg.seed)
print(DEV, asdict(cfg))

## Data

Tahoe-100M subset: 25 cell lines × 100 drugs at three doses, plus DMSO controls; 991,899 cells.
Each cell is 2000 highly variable genes reduced to 50 principal components, with normalisation,
gene selection, scaling and PCA all fitted on the training split only
(`notebooks/tahoe/tahoe_colab_downloader_with_pca.ipynb`). The split holds out about 20% of
(cell line, drug) **pairs**: every validation cell comes from a combination never seen in
training, while its line and its drug each appear in other training pairs. So validation loss
here already measures generalisation to unseen combinations. `x` is the 50-vector scaled so
its overall variance is 1.

The condition is the (cell line, drug) pair by default, 2525 ids, which the backbone turns into
a drug embedding plus a cell-line embedding, so an unseen pair is still a seen drug and a seen
line. Its text is the drug's mechanism of action and targets plus the cell line's name and
organ, e.g. "Binimetinib, mek inhibitor targeting MAP2K1, MAP2K2, in HepG2/C3A liver cells",
written from the public metadata before any result.

In [ ]:
from guidance.data.cells import make_loaders

train_loader, val_loader, meta = make_loaders(
    batch_size=cfg.batch_size, seed=cfg.split_seed, condition=cfg.condition,
    dose=None if cfg.dose < 0 else cfg.dose, limit=cfg.limit)

xb, cb = next(iter(train_loader))
print("x", tuple(xb.shape), f"rms {float(xb.pow(2).mean().sqrt()):.2f}", "| cond", tuple(cb.shape))
print(f"train cells {meta['n_train']:,} | val cells {meta['n_val']:,} | "
      f"conditions {meta['n_conditions']} ({meta['condition']}) | split: {meta['split']}")
print("example text:", meta["cond_text"][1])

## The objective: a per-drug Gaussian classifier on clean cell states

No pretrained classifier exists for this label set and the project trains none, so the
classifier is fitted by moments: one mean per drug over its training cells, one shared
covariance from the within-drug residuals (plus a small diagonal shrinkage), class frequencies
as priors. That is linear discriminant analysis, and its log-posterior
$\log p_r(y \mid x) = \log\operatorname{softmax}_d(-\tfrac12(x-\mu_d)^\top\Sigma^{-1}(x-\mu_d) + \log\pi_d)_y$
is differentiable in $x$, which is all TFG needs. The target is the drug; the drug of each cell
comes from the loader's condition table.

The **scorer** is a different function on the same labels: a $k$-nearest-neighbour vote over
real training cells with their true drug. The check below reports both scorers' accuracy on
real validation cells. Drug effects in expression are often small next to cell-line
differences, so expect these ceilings to be low; they bound what any guidance can reach here,
and the validation cells are unseen line-drug pairs.

In [ ]:
from guidance.data.cells import condition_texts

# Drug of every condition id, from the loader's table (the backbone cell builds its own copy later).
COND_DRUG = meta["cond_ids_table"][:, 0].to(DEV)
X_TRAIN = train_loader.dataset.x.to(DEV); Y_TRAIN = COND_DRUG[train_loader.dataset.cond.to(DEV)]   # drug of each cell
X_VAL = val_loader.dataset.x.to(DEV);     Y_VAL = COND_DRUG[val_loader.dataset.cond.to(DEV)]
N_DRUGS, D = meta["n_drugs"], X_TRAIN.shape[1]
_drug_text, _ = condition_texts()
DRUG_TEXT = [_drug_text[n] for n in meta["drug_names"]]

# Fitted by moments on the training cells. Nothing is trained.
COUNTS = torch.bincount(Y_TRAIN, minlength=N_DRUGS).float()
MU = torch.zeros(N_DRUGS, D, device=DEV).index_add_(0, Y_TRAIN, X_TRAIN) / COUNTS.clamp(min=1)[:, None]
RES = X_TRAIN - MU[Y_TRAIN]
SIGMA = RES.T @ RES / len(RES) + cfg.shrinkage * torch.eye(D, device=DEV)
PREC = torch.linalg.inv(SIGMA)
LOG_PRIOR = torch.log(COUNTS.clamp(min=1) / COUNTS.sum())


def guide_logits(x):
    '''Unnormalised log p(y | x) of the Gaussian classifier, for every drug. Differentiable in x.'''
    d = x[:, None, :] - MU[None]                              # (B, N_DRUGS, D)
    return -0.5 * torch.einsum("bkd,de,bke->bk", d, PREC, d) + LOG_PRIOR


def lda_label(x):
    return guide_logits(x).argmax(1)


# Independent scorer: the shared k-nearest-neighbour drug scorer over real training cells.
# Same labels, different decision function, never seen by the sampler.
from guidance.eval.cell_metrics import all_metrics, KNNConditionScorer
SCORER = KNNConditionScorer(X_TRAIN, Y_TRAIN, k=25, seed=cfg.seed, device=DEV)
knn_label = SCORER.predict
KNN_K, KNN_REF = SCORER.k, SCORER.x


# Ceilings: how often each scorer recognises the drug of a REAL validation cell. These bound
# what guidance toward a drug can reach on this data, and they are on unseen line-drug pairs.
n_check = min(20_000, len(X_VAL))
LDA_REAL_ACC = (lda_label(X_VAL[:n_check]) == Y_VAL[:n_check]).float().mean().item()
KNN_REAL_ACC = (knn_label(X_VAL[:n_check]) == Y_VAL[:n_check]).float().mean().item()
print(f"Gaussian classifier: {N_DRUGS} drugs, shared {D}x{D} covariance, fitted on {len(X_TRAIN):,} training cells")
print(f"accuracy on {n_check:,} real validation cells: Gaussian classifier {LDA_REAL_ACC:.1%} | "
      f"{KNN_K}-NN scorer {KNN_REAL_ACC:.1%} | chance {1 / N_DRUGS:.1%}")
if KNN_REAL_ACC < 3 / N_DRUGS:
    print("the scorer barely recognises real drug cells; guidance accuracy below will be near chance too. "
          "Report it, and read the per-drug MMD instead.")

## Backbone

In [ ]:
def timestep_embedding(t, dim):
    half = dim // 2
    freqs = torch.exp(-math.log(10_000.0) * torch.arange(half, device=t.device) / half)
    args = t.float()[:, None] * freqs[None, :]
    return torch.cat([args.cos(), args.sin()], dim=-1)

# ---- The model of notebooks/tahoe/Tahoe_PCA_Diffusion_CFG.ipynb, a teammate's classifier-free-
# guidance base for these cells, kept verbatim: drug and cell line are embedded separately and
# summed with the time embedding into one vector that enters every residual block.
class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        half = dim // 2
        freqs = torch.exp(-math.log(10000.0) * torch.arange(half) / max(half - 1, 1))
        self.register_buffer("freqs", freqs, persistent=False)

    def forward(self, t):
        args = t.float()[:, None] * self.freqs[None, :].to(t.device)
        return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)


class ResidualBlock(nn.Module):
    def __init__(self, dim, cond_dim):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.fc1 = nn.Linear(dim + cond_dim, dim)
        self.fc2 = nn.Linear(dim, dim)
        self.act = nn.SiLU()

    def forward(self, x, cond):
        h = torch.cat([self.norm(x), cond], dim=-1)
        h = self.fc2(self.act(self.fc1(h)))
        return x + h


class DiffusionMLP(nn.Module):
    def __init__(self, data_dim, n_drugs, n_lines, time_dim=128, cond_dim=128, hidden_dim=256, n_blocks=4):
        super().__init__()
        self.data_dim = data_dim
        self.time_embed = SinusoidalTimeEmbedding(time_dim)
        self.time_mlp = nn.Sequential(nn.Linear(time_dim, cond_dim), nn.SiLU(), nn.Linear(cond_dim, cond_dim))
        self.drug_embed = nn.Embedding(n_drugs + 1, cond_dim)   # last row is the null/unconditional id
        self.line_embed = nn.Embedding(n_lines + 1, cond_dim)   # last row is the null/unconditional id
        self.input_proj = nn.Linear(data_dim, hidden_dim)
        self.blocks = nn.ModuleList([ResidualBlock(hidden_dim, cond_dim) for _ in range(n_blocks)])
        self.output_proj = nn.Linear(hidden_dim, data_dim)

    def forward(self, x, t, drug_id, line_id):
        cond = self.time_mlp(self.time_embed(t)) + self.drug_embed(drug_id) + self.line_embed(line_id)
        h = self.input_proj(x)
        for block in self.blocks:
            h = block(h, cond)
        return self.output_proj(h)


# ---- Our wrapper. The notebooks pass one cond_id and t in [0, 1]. The loader's table turns the
# id into (drug_id, line_id); the extra last row is the null condition, both ids null, which is
# their joint dropout. t is scaled back to the integer step their time embedding was written for.
COND_TABLE = torch.cat([meta["cond_ids_table"],
                        torch.tensor([[meta["n_drugs"], meta["n_lines"]]])]).to(DEV)
assert int(COND_TABLE.min()) >= 0, "this backbone needs condition = 'pair' or 'drug'"


class CellDiffusionMLP(DiffusionMLP):
    def __init__(self):
        super().__init__(meta["shape"][0], n_drugs=meta["n_drugs"], n_lines=meta["n_lines"])
        self.null = meta["n_conditions"]

    def forward(self, x, t, cond_ids=None):
        if cond_ids is None:
            cond_ids = torch.full((x.shape[0],), self.null, device=x.device, dtype=torch.long)
        ids = COND_TABLE[cond_ids]
        return super().forward(x, t * cfg.K, ids[:, 0], ids[:, 1])


def make_backbone():
    return CellDiffusionMLP()

# Only the trunk is adapted by TC-LoRA: the input projection and the residual blocks' two
# linear layers each. Time MLP, the two embeddings and the output projection stay untouched.
SKIP_NAMES = ("output_proj", "time_mlp", "drug_embed", "line_embed")

torch.manual_seed(cfg.seed)      # model init and training randomness follow `seed`
net = make_backbone().to(DEV)
print(f"backbone params: {sum(p.numel() for p in net.parameters()):,}")

## Diffusion — cosine schedule, epsilon prediction, DDIM sampler

In [ ]:
def cosine_alpha_bar(K, s=0.008, device="cpu"):
    steps = torch.arange(K + 1, dtype=torch.float32, device=device) / K
    f = torch.cos((steps + s) / (1 + s) * math.pi / 2) ** 2
    return (f / f[0])[1:].clamp(1e-5, 0.9999)

ABAR = cosine_alpha_bar(cfg.K, device=DEV)
NULL_ID = meta["n_conditions"]
CLIP_X0 = 0.0                                          # no clamp, as their scheduler (clip_sample=False)
print("x0 clamp: off")


def add_noise(x0, gen=None):
    n = x0.shape[0]
    k = torch.randint(0, cfg.K, (n,), device=x0.device, generator=gen)
    eps = torch.randn(x0.shape, device=x0.device, generator=gen)
    ab = ABAR[k].view(n, *([1] * (x0.dim() - 1)))
    return ab.sqrt() * x0 + (1 - ab).sqrt() * eps, k, eps


def drop_labels(y, p_drop=0.1):
    keep = torch.rand(y.shape[0], device=y.device) >= p_drop
    return torch.where(keep, y, torch.full_like(y, NULL_ID))


@torch.no_grad()
def ddim_sample(eps_fn, n, shape, steps, clip_x0=None, x_init=None, eta=None, generator=None):
    '''Plain deterministic DDIM, TFG's convention: clip x0, then RECOMPUTE eps from the
    clipped x0 before stepping. Keeping the original eps is a different update whenever
    the clamp engages; the two can differ by O(1) on real images. The TFG sampler in
    src/ reduces to exactly this at rho = mu = 0, and the cell below asserts it.'''
    clip_x0 = CLIP_X0 if clip_x0 is None else clip_x0
    idx = torch.linspace(cfg.K - 1, 0, steps).long().to(DEV)
    eta = cfg.eta if eta is None else eta
    x = torch.randn(n, *shape, device=DEV, generator=generator) if x_init is None else x_init.clone()
    for i, k in enumerate(idx):
        t = (k.float() / cfg.K).repeat(n)
        eps = eps_fn(x, t)
        ab = ABAR[k]
        ab_prev = ABAR[idx[i + 1]] if i + 1 < len(idx) else torch.ones_like(ab)
        x0 = (x - (1 - ab).sqrt() * eps) / ab.sqrt()
        if clip_x0:
            x0 = x0.clamp(-clip_x0, clip_x0)
            eps = (x - ab.sqrt() * x0) / (1 - ab).sqrt()      # recompute from the clipped x0
        # eta = 0 is deterministic DDIM; eta = 1 is ancestral DDPM (DDIM paper eq. 16), with fresh
        # noise at every step drawn from `generator` so paired rows stay paired.
        sigma = eta * ((1 - ab_prev) / (1 - ab) * (1 - ab / ab_prev)).sqrt()
        x = ab_prev.sqrt() * x0 + (1 - ab_prev - sigma ** 2).clamp(min=0).sqrt() * eps
        if eta > 0:
            x = x + sigma * torch.randn(x.shape, device=DEV, generator=generator)
    return x

## Base model — train, or load `base.pt`

Identical to the other arms. Upload `base.pt` from a previous run to skip the eight
minutes; every arm then sits on the same frozen weights.

In [ ]:
VAL_SEED = cfg.seed + 999


def run_epochs(model, params, n_epochs, lr, tag, train_fn, val_fn, ckpt):
    opt = torch.optim.AdamW(params, lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(n_epochs, 1))
    hist, best, best_ep = [], float("inf"), 0
    for ep in range(1, n_epochs + 1):
        model.train(); tot = num = 0; t0 = time.time()
        for x, y in train_loader:
            x, y = x.to(DEV), y.to(DEV)
            loss = train_fn(x, y)
            opt.zero_grad(set_to_none=True); loss.backward()
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            opt.step()
            tot += loss.item() * x.shape[0]; num += x.shape[0]
        sched.step()
        val = validate(model, val_fn)
        is_best = val < best
        if is_best:
            best, best_ep = val, ep
            torch.save({"state": model.state_dict(), "epoch": ep, "val": val}, ckpt)
        hist.append({"epoch": ep, "train": tot / num, "val": val, "best": is_best,
                     "seconds": time.time() - t0})
        print(f"{tag} epoch {ep:3d}  train {tot/num:.4f}  val {val:.4f}"
              f"{'  *best' if is_best else ''}  ({time.time()-t0:.0f}s)")
    print(f"{tag}: best val {best:.4f} at epoch {best_ep}, saved to {ckpt}")
    return hist


@torch.no_grad()
def validate(model, val_fn):
    model.eval()
    gen = torch.Generator(device=DEV).manual_seed(VAL_SEED)
    tot = num = 0
    for x, y in val_loader:
        x, y = x.to(DEV), y.to(DEV)
        tot += val_fn(x, y, gen).item() * x.shape[0]; num += x.shape[0]
    return tot / num


def base_train(x, y):
    xk, k, eps = add_noise(x)
    return F.mse_loss(net(xk, k.float() / cfg.K, drop_labels(y)), eps)


def base_val(x, y, gen):
    xk, k, eps = add_noise(x, gen)
    return F.mse_loss(net(xk, k.float() / cfg.K, y), eps)


CKPT_META = {"split_seed": cfg.split_seed, "n_conditions": meta["n_conditions"],
             "shape": list(meta["shape"]), "K": cfg.K, "schedule": "cosine",
             "backbone_params": sum(p.numel() for p in net.parameters())}

if cfg.reuse_base and os.path.exists("base.pt"):
    ck = torch.load("base.pt", map_location=DEV)
    saved = ck.get("meta")
    if saved is None:
        print("WARNING: base.pt carries no setup metadata (older run); split and schedule "
              "cannot be verified.")
    else:
        mismatch = {k: (saved.get(k), v) for k, v in CKPT_META.items() if saved.get(k) != v}
        assert not mismatch, f"base.pt was trained under a different setup: {mismatch}"
    net.load_state_dict(ck["state"])
    base_hist = [{"epoch": ck["epoch"], "train": float("nan"), "val": ck["val"], "best": True}]
    print(f"loaded base.pt (epoch {ck['epoch']}, val {ck['val']:.4f}); skipping base training")
else:
    base_hist = run_epochs(net, list(net.parameters()), cfg.base_epochs, cfg.lr_base,
                           "base", base_train, base_val, "base.pt")
    ck = torch.load("base.pt", map_location=DEV); ck["meta"] = CKPT_META; torch.save(ck, "base.pt")

# Whichever branch ran, sample from the BEST checkpoint on disk, not the last epoch left in
# `net`. Without this a fresh run and a reused-checkpoint run would use different weights
# while the manifest reported the same best validation loss for both.
ck = torch.load("base.pt", map_location=DEV)
net.load_state_dict(ck["state"])
base_frozen = net.eval()
for p in base_frozen.parameters():
    p.requires_grad_(False)
print(f"sampling from base.pt epoch {ck['epoch']} (val {ck['val']:.4f})")

## Set up TFG

TFG expects an **unconditional** denoiser. Ours is (drug, cell line)-conditional with a null
condition, so `eps_uncond` runs it with both ids null throughout — the label path is off, and the
classifier gradient is the only route by which the drug enters.

The objective is `log p(y | x̂₀)` from the Gaussian classifier, evaluated on the predicted clean
cell state. That is the role their `ImageLabelGuidance` plays.

First check: at `rho = mu = 0` the TFG sampler must reproduce plain DDIM to numerical
precision. If it does not, nothing downstream can be trusted.

In [ ]:
def eps_uncond(x, t):
    # TFG assumes an unconditional denoiser. Ours is class-conditional with a null label, so the
    # null label stands in for "unconditional" and the classifier gradient is the only way in.
    return base_frozen(x, t, torch.full((x.shape[0],), NULL_ID, device=x.device, dtype=torch.long))


def make_log_p(y):
    '''log p(y | x0) under the Gaussian classifier, for a fixed batch of target drugs y.
    This is the role of ImageLabelGuidance: the quantity Algorithm 1 climbs at every step.'''
    def log_p(x0):
        logits = guide_logits(x0)
        yy = y.repeat(x0.shape[0] // y.shape[0])          # smoothing may stack eps_bsz copies
        return F.log_softmax(logits, dim=-1).gather(1, yy[:, None]).squeeze(1)
    return log_p


tfg_cfg = TFGConfig(rho=cfg.rho, mu=cfg.mu, sigma=cfg.sigma, eps_bsz=cfg.eps_bsz,
                    iter_steps=cfg.iter_steps, recur_steps=cfg.recur_steps,
                    clip_x0=CLIP_X0, eta=cfg.eta)

# sanity: guidance off == plain DDIM, same starting noise
g = torch.Generator(device=DEV).manual_seed(0)
x0_probe = torch.randn(8, *meta["shape"], device=DEV, generator=g)
y_probe = torch.arange(8, device=DEV)
off = TFG(eps_uncond, make_log_p(y_probe), ABAR, TFGConfig(rho=0, mu=0, iter_steps=0, clip_x0=CLIP_X0))
a = off.sample(8, meta["shape"], 10, x_init=x0_probe)
b = ddim_sample(eps_uncond, 8, meta["shape"], 10, x_init=x0_probe, eta=0.0)   # deterministic on both sides
assert torch.allclose(a, b, atol=1e-4), (a - b).abs().max().item()
assert off.stats.objective_evals == 0 and off.stats.denoiser_backward == 0
print("rho = mu = 0 reproduces plain DDIM exactly")
print("TFG config:", asdict(tfg_cfg))

## Sample — three rows from the same starting noise, five drugs

| row | what | guidance |
|---|---|---|
| **uncond** | base, null condition | none |
| **TFG** | base, null condition | Gaussian-classifier gradient toward drug `y` |
| **TFG, wrong y** | base, null condition | gradient toward the next drug in the list |

The five treated drugs with the most validation cells, `samples_per_condition` each, all from
the same starting noise. Real cells of a drug are pooled over cell lines, and so are the
samples, since the target here is the drug alone. The last row is the diagnostic: if TFG works,
its samples should score as the wrong drug, not `y`. Every row is timed; TFG costs a denoiser
backward pass and several objective evaluations per step, so the breakdown is reported rather
than one NFE. **Paired means paired:** every TFG call gets its own generator seeded identically,
so the only thing that differs between rows is the target.

In [ ]:
# Five treated drugs with the most validation cells, PER samples each; every metric is per drug.
counts = torch.bincount(Y_VAL, minlength=N_DRUGS); counts[0] = 0                  # drop DMSO
EVAL_DRUGS = counts.topk(5).indices.tolist()
PER = cfg.samples_per_condition
y_eval = torch.tensor(EVAL_DRUGS, device=DEV).repeat_interleave(PER)
y_wrong = y_eval.roll(PER)                        # each block gets the next drug as target
N = len(y_eval)
print("drugs:", [DRUG_TEXT[c].split(",")[0] for c in EVAL_DRUGS])
g = torch.Generator(device=DEV).manual_seed(cfg.seed + 1)
X0 = torch.randn(N, *meta["shape"], device=DEV, generator=g)

def paired_gen():
    '''Fresh generator, same seed: identical internal randomness for every matched call.'''
    return torch.Generator(device=DEV).manual_seed(cfg.seed + 1234)

def timed(fn):
    if DEV.type == "cuda": torch.cuda.synchronize()
    t0 = time.time(); out = fn()
    if DEV.type == "cuda": torch.cuda.synchronize()
    return out, (time.time() - t0) / N

rows = {}
rows["uncond"], t_uncond = timed(lambda: ddim_sample(eps_uncond, N, meta["shape"], cfg.sample_steps, x_init=X0, generator=paired_gen()))
tfg = TFG(eps_uncond, make_log_p(y_eval), ABAR, tfg_cfg)    # Algorithm 1, their CIFAR-10 settings
rows["tfg"], t_tfg = timed(lambda: tfg.sample(N, meta["shape"], cfg.sample_steps, x_init=X0, generator=paired_gen()))
tfg_stats = tfg.stats
tfg_w = TFG(eps_uncond, make_log_p(y_wrong), ABAR, tfg_cfg)
rows["tfg_wrong"], _ = timed(lambda: tfg_w.sample(N, meta["shape"], cfg.sample_steps, x_init=X0, generator=paired_gen()))

# How often the guided clean estimate hit the clamp -- the objective's gradient is zero there.
SATURATION = {k: (v.abs() >= 0.999 * CLIP_X0).float().mean().item() for k, v in rows.items()}

print(f"{'row':>10} {'s / sample':>11}   denoiser fwd / bwd   objective calls   saturated px")
print(f"{'uncond':>10} {t_uncond:>11.3f}   {cfg.sample_steps:>4} / {0:<4}          0          {SATURATION['uncond']:6.1%}")
print(f"{'tfg':>10} {t_tfg:>11.3f}   {tfg_stats.denoiser_evals:>4} / {tfg_stats.denoiser_backward:<4}          "
      f"{tfg_stats.objective_evals}        {SATURATION['tfg']:6.1%}")

## Condition accuracy — scored by the *other* function

The k-nearest-neighbour vote never touched the sampler. Its accuracy on each row is the target
metric, restricted to the five drugs sampled. For the wrong-target row the number reported is
accuracy against the drug that was pushed toward: high means guidance followed its target. Read
these against the ceilings on real cells printed above, not against 100%.

In [ ]:
@torch.no_grad()
def acc_of(x, y):
    return (knn_label(x) == y).float().mean().item()

ACC = {
    "real_val_cells_knn": KNN_REAL_ACC,
    "real_val_cells_gaussian": LDA_REAL_ACC,
    "uncond": acc_of(rows["uncond"], y_eval),
    "tfg": acc_of(rows["tfg"], y_eval),
    "tfg_wrong_vs_target": acc_of(rows["tfg_wrong"], y_wrong),
    "tfg_wrong_vs_original": acc_of(rows["tfg_wrong"], y_eval),
}
for k, v in ACC.items():
    print(f"{k:>26}: {v:6.1%}")
print()
print(f"chance = {1 / N_DRUGS:.1%}.")
print("If guidance follows its target, 'tfg' sits above 'uncond' and 'tfg_wrong_vs_target' is comparably high.")
if ACC["tfg"] < ACC["uncond"] + 0.02:
    print("little steering at these settings: try larger rho/mu or more iter_steps.")
print()
print("Accuracy alone rewards collapsed samples even with a separate scorer. Read it with the per-drug MMD and variance ratio below.")

## Samples

In [ ]:
import matplotlib.pyplot as plt

real_x = X_VAL.cpu().numpy(); real_y = Y_VAL.cpu().numpy()
cmap = plt.get_cmap("tab10")
fig, ax = plt.subplots(1, 4, figsize=(19, 4.4), sharex=True, sharey=True)
panels = [("real (val)", real_x, real_y), ("uncond", rows["uncond"].cpu().numpy(), y_eval.cpu().numpy()),
          ("TFG, target y", rows["tfg"].cpu().numpy(), y_eval.cpu().numpy()),
          ("TFG, wrong target", rows["tfg_wrong"].cpu().numpy(), y_wrong.cpu().numpy())]
for a, (name, pts, cs) in zip(ax, panels):
    for j, c in enumerate(EVAL_DRUGS):
        p = pts[cs == c][:PER]
        a.scatter(p[:, 0], p[:, 1], s=6, alpha=0.5, color=cmap(j),
                  label=DRUG_TEXT[c].split(",")[0] if a is ax[0] else None)
    a.set_title(name); a.set_xlabel("PC1")
ax[0].set_ylabel("PC2"); ax[0].legend(fontsize=7, markerscale=2); plt.tight_layout(); plt.show()

## Guidance-strength sweep

TFG's `rho` and `mu` are the knobs. A short sweep at fixed `mu/rho` ratio shows the
accuracy-vs-time trade-off. This is the analogue of a classifier-free-guidance strength
sweep, and it is what a learned guidance controller has to beat: the **best point on this
curve**, not the default.

This is a first pass, not TFG's own hyperparameter search. Pick a setting here, then report it with fresh sampling seeds — do not report the
maximum of the sweep as the result.

In [ ]:
SWEEP = list(cfg.sweep)
sweep_rows = []
for scale in SWEEP:
    c = TFGConfig(rho=cfg.rho * scale, mu=cfg.mu * scale, sigma=cfg.sigma, eps_bsz=cfg.eps_bsz,
                  iter_steps=cfg.iter_steps if scale > 0 else 0, recur_steps=cfg.recur_steps,
                  clip_x0=CLIP_X0, eta=cfg.eta)
    s, dt = timed(lambda: TFG(eps_uncond, make_log_p(y_eval), ABAR, c).sample(
        N, meta["shape"], cfg.sample_steps, x_init=X0, generator=paired_gen()))
    sweep_rows.append({"scale": scale, "rho": c.rho, "mu": c.mu, "acc": acc_of(s, y_eval), "s_per_sample": dt})
    print(f"scale {scale:>4}  rho {c.rho:>5.2f}  mu {c.mu:>5.3f}  acc {sweep_rows[-1]['acc']:6.1%}  {dt:.3f}s/sample")

fig, ax = plt.subplots(figsize=(5.5, 3.6))
ax.plot([r["scale"] for r in sweep_rows], [r["acc"] for r in sweep_rows], marker="o")
ax.axhline(KNN_REAL_ACC, ls="--", c="grey", lw=1, label=f"real cells, kNN scorer ({KNN_REAL_ACC:.1%})")
ax.axhline(1 / N_DRUGS, ls=":", c="grey", lw=1, label="chance")
ax.set_xlabel("guidance scale (x default rho, mu)"); ax.set_ylabel(f"accuracy ({KNN_K}-NN scorer)")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

## Per-drug quality and diversity

Accuracy says whether samples are recognised as the requested drug; it says nothing about
whether they look like real cells of that drug. For each of the five drugs, from the shared
metrics module: MMD, energy distance, Fréchet distance on the PCs, precision and recall between
the samples requested for that drug and its real validation cells (pooled over cell lines), the
variance ratio for collapse, and the kNN drug accuracy. The wrong-target row is
scored against the drug that was requested, so if guidance moved the samples it is worse than
the correct row.

In [ ]:
# Shared module src/guidance/eval/cell_metrics.py, per drug against its real validation cells
# (pooled over cell lines): MMD, energy distance, Frechet distance on the PCs, precision, recall,
# variance ratio, and the kNN drug accuracy. The wrong-target row is scored against the drug
# that was requested.
gen_y = y_eval
METRICS = {"eval_drugs": EVAL_DRUGS, "eval_drug_text": [DRUG_TEXT[c] for c in EVAL_DRUGS], "samples_per_condition": PER,
           "scorer": SCORER.describe(), "scorer_real_val_drug_accuracy": KNN_REAL_ACC}
print(f"{'drug':30s} {'arm':10s} {'MMD':>6} {'energy':>7} {'FD_pc':>7} {'prec':>5} {'rec':>5} {'var':>6} {'acc':>6} {'n real':>6}")
for name in ("uncond", "tfg", "tfg_wrong"):
    per = []
    for c in EVAL_DRUGS:
        r = X_VAL[Y_VAL == c]; gsel = rows[name][gen_y == c]
        m = all_metrics(gsel, r, seed=cfg.seed)
        m["cond_acc"] = SCORER.accuracy(gsel, torch.full((len(gsel),), c, device=DEV))
        per.append({"drug": int(c), "text": DRUG_TEXT[c], **m})
        print(f"{DRUG_TEXT[c][:30]:30s} {name:10s} {m['mmd']:6.3f} {m['energy']:7.3f} {m['frechet_pc']:7.2f} "
              f"{m['precision']:5.2f} {m['recall']:5.2f} {m['var_ratio']:6.2f} {m['cond_acc']:6.1%} {len(r):6d}")
    METRICS[f"{name}_per_drug"] = per
    for k in ("mmd", "energy", "frechet_pc", "precision", "recall", "var_ratio", "cond_acc"):
        METRICS[f"{name}_{k}_mean"] = float(np.mean([p_[k] for p_ in per]))
FID_SCORES = {}

## Manifest

In [ ]:
report = {
    "method": "TFG training-free guidance (arXiv:2409.15761), per-drug Gaussian classifier on cell states",
    "modality": "cells",
    "code_version": GIT_SHA,
    "config": asdict(cfg),
    "tfg_config": asdict(tfg_cfg),
    "guide_classifier": f"Gaussian classifier (LDA), {N_DRUGS} drugs, shared covariance, shrinkage {cfg.shrinkage}, fitted on {len(X_TRAIN):,} training cells",
    "eval_classifier": f"{KNN_K}-nearest-neighbour vote over {len(KNN_REF):,} real training cells with true drug labels",
    "label_path": False,
    "accuracy": ACC,
    "n_eval_samples": N,
    "saturated_pixel_fraction": SATURATION,
    "sweep": sweep_rows,
    "seconds_per_sample": {"uncond": t_uncond, "tfg": t_tfg},
    "tfg_cost_per_sample": {"denoiser_forward": tfg_stats.denoiser_evals,
                            "denoiser_backward": tfg_stats.denoiser_backward,
                            "objective_evals": tfg_stats.objective_evals},
    "plain_ddim_cost_per_sample": {"denoiser_forward": cfg.sample_steps},
    "base_best_val": min(h["val"] for h in base_hist),
    "checkpoint_meta": CKPT_META,
    "trainable_params": 0,
    "frozen_backbone": sum(p.numel() for p in base_frozen.parameters()),
    "nfe_sample_steps": cfg.sample_steps,
    "train_seconds": {"base": sum(h.get("seconds", 0) for h in base_hist)},
    "hardware": torch.cuda.get_device_name(0) if DEV.type == "cuda" else "cpu",
    "device": str(DEV),
}
report.update(globals().get("FID_SCORES", {}))
report.update(globals().get("METRICS", {}))
print(json.dumps(report, indent=2))
with open("manifest_tfg_cells.json", "w") as f:
    json.dump(report, f, indent=2)

## Before reporting any of this

- [ ] Both scorers' accuracy on real validation cells reported next to every generated number:
      they are the ceiling, and they are on unseen (cell line, drug) pairs
- [ ] `tfg_wrong_vs_target` is comparably high to `tfg`: guidance follows its target
- [ ] MMD and energy distance read together with accuracy and the variance ratio, never accuracy alone
- [ ] Three **sampling** seeds on one checkpoint, mean +/- std, stated as such
- [ ] Guidance settings chosen on the sweep, then reported with fresh seeds
- [ ] Time per sample and the denoiser/objective call counts reported next to accuracy
- [ ] Saturation fraction reported; the objective is log p(y | clamp(x0))
- [ ] `condition`, `dose` and `shrinkage` stated next to every number
- [ ] Described as TFG's `guide_step` **re-expressed** against our sampler, their CIFAR-10
      hyperparameters as starting settings (not tuned for cell states), `eta = 0`, a Gaussian
      classifier fitted by moments in place of a pretrained network, on a null-condition
      conditional model standing in for an unconditional one